https://unit8co.github.io/darts/examples/07-NBEATS-examples.html

In [1]:
import torch
import numpy as np
import pandas as pd
import shutil

from darts import TimeSeries
from darts.models import NBEATSModel
from darts.dataprocessing.transformers import Scaler, MissingValuesFiller
from darts.metrics import mape, r2_score, mae, rmse
from darts import concatenate

import matplotlib.pyplot as plt
import plotly.graph_objects as go

import warnings

warnings.filterwarnings("ignore")
import logging

logging.disable(logging.CRITICAL)

In [2]:
def display_forecast(pred_series, ts_transformed, start_date=None):
    plt.figure(figsize=(8, 5))
    if start_date:
        ts_transformed = ts_transformed.drop_before(start_date)
    ts_transformed.univariate_component(0).plot(label="actual")
    pred_series.plot(label=("predicted"))
    plt.title(
        "R2: {}\n".format(r2_score(ts_transformed.univariate_component(0), pred_series))
        + "MAPE: {}\n".format(mape(ts_transformed.univariate_component(0), pred_series))
        + "MAE: {}\n".format(mae(ts_transformed.univariate_component(0), pred_series))
        + "RMSE: {}\n".format(rmse(ts_transformed.univariate_component(0), pred_series))
    )
    plt.legend()

def display_forecast_plotly(title_text, pred_series, ts_transformed, start_date=None):

    if start_date:
        ts_transformed = ts_transformed.drop_before(start_date)

    fig = go.Figure()

    fig.add_trace(go.Scatter(x=ts_transformed.univariate_component(0).time_index, y=ts_transformed.univariate_component(0).pd_series(), name='actual'))
    fig.add_trace(go.Scatter(x=pred_series.time_index, y=pred_series.pd_series(), name='predicted'))

    # add title with R2, MAPE, MAE and RMSE
    fig.update_layout(title=f"{title_text}<br>R2: {r2_score(ts_transformed.univariate_component(0), pred_series)} | "
        + f"MAPE: {mape(ts_transformed.univariate_component(0), pred_series)} | "
        + f"MAE: {mae(ts_transformed.univariate_component(0), pred_series)} | "
        + f"RMSE: {rmse(ts_transformed.univariate_component(0), pred_series)}")

    fig.show()

## Get the Data & Process (merge)

In [3]:
# Load data
# df = pd.read_csv('../data/01-output-BTCUSDT_1d-from-2020-12-31 00:00:00-until-2022-12-31 00:00:00-log-return.csv')
df = pd.read_csv('../data/01-output-ETHUSDT_1d-from-2018-12-31 00:00:00-until-2020-12-31 00:00:00-log-return.csv')
# df = pd.read_csv('../data/01-output-SOLUSDT_1d-from-2020-12-31 00:00:00-until-2022-12-31 00:00:00-log-return.csv')


data_name = 'ETHUSDT_1h' # BTCUSDT_1h, ETHUSDT_1h, SOLUSDT_1h
from_date = '2019-05-31' # 2021-11-30, 2019-05-31, 2021-03-31
# until_date = '2019-05-31'
time_delay = 168 # 72, 168
time_delay_in_days = int(time_delay / 24)
slice_size = 336 # 336, 504, 720, 1080
slice_size_in_days = int(slice_size / 24)

target_feature = 'processed_log_return_wtmra_0'

# df_features = pd.read_csv(f'../data/log-returns-{target_feature}-features_{data_name}_time_delay_{time_delay}_slice_size_{slice_size}.csv')
df_features = pd.read_csv(f'../results/processed_log_return_wtmra_0_{data_name}_time_delay_{time_delay}_slice_size_{slice_size}/features/features_{data_name}_time_delay_{time_delay}_slice_size_{slice_size}.csv')

In [4]:
# Format end_date to remove time
df_features['end_date'] = df_features['end_date'].apply(lambda x: x.split(' ')[0])

# inner join df (date) and df_features (end_date)
df_merged = df.merge(df_features.rename(columns={'end_date': 'date'}), on='date', how='inner')

In [5]:
# Filter from_date 
df_merged = df_merged[df_merged['date'] >= from_date]
df_merged

,date,open,high,low,close,volume,original_close,processed_log_return,outliers_processed_log_return,normalized_outliers_processed_log_return,...,start_date,connected_components_entropy,loops_entropy,voids_entropy,connected_components_amplitude,loops_amplitude,voids_amplitude,connected_components_number_of_points,loops_number_of_points,voids_number_of_points
137,2019-05-31,268.92,288.62,240.14,254.56,1.066279e+06,254.56,-0.054543,-0.054543,-0.622462,...,2019-05-17,7.037392,4.543693,-1.0,0.029415,0.007732,0.000000,167,35,0
138,2019-06-01,254.59,268.72,245.21,267.90,6.021539e+05,267.90,0.051077,0.051077,0.490950,...,2019-05-18,6.982006,4.645676,-1.0,0.034858,0.005502,0.000000,167,39,0
139,2019-06-02,267.90,275.50,260.68,264.33,4.556770e+05,264.33,-0.013415,-0.013415,-0.188912,...,2019-05-19,7.049736,4.659735,-1.0,0.032048,0.007479,0.000000,167,37,0
140,2019-06-03,264.33,273.20,263.20,268.88,3.015362e+05,268.88,0.017067,0.017067,0.132423,...,2019-05-20,7.002072,4.528437,-1.0,0.032048,0.006023,0.000000,167,37,0
141,2019-06-04,268.87,270.00,248.00,249.91,3.973607e+05,249.91,-0.073164,-0.073164,-0.818766,...,2019-05-21,7.006916,4.343220,-1.0,0.032048,0.009122,0.000000,167,32,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
713,2020-12-27,626.78,652.91,615.26,637.44,9.585855e+05,637.44,0.016801,0.016801,0.129618,...,2020-12-13,6.915810,4.848597,-1.0,0.049243,0.007238,0.000000,167,42,0
714,2020-12-28,637.44,717.13,625.00,685.11,1.859968e+06,685.11,0.072119,0.072119,0.712768,...,2020-12-14,6.959831,4.958643,-1.0,0.049243,0.008051,0.000000,167,43,0
715,2020-12-29,685.10,748.09,681.04,730.41,1.627154e+06,730.41,0.064027,0.064027,0.627458,...,2020-12-15,7.002490,4.883499,-1.0,0.049243,0.007934,0.000000,167,40,0
716,2020-12-30,730.40,740.78,689.20,732.00,1.106876e+06,732.00,0.002174,0.002174,-0.024568,...,2020-12-16,7.019302,5.131655,0.0,0.049243,0.007113,0.000042,167,47,1


In [6]:
df_merged[target_feature] = df_merged[target_feature].astype('float32')
# df_merged['processed_log_return_wtmra_5_4_3_2_1'] = df_merged['processed_log_return_wtmra_5_4_3_2_1'].astype('float32')
df_merged['connected_components_entropy'] = df_merged['connected_components_entropy'].astype('float32')
df_merged['loops_entropy'] = df_merged['loops_entropy'].astype('float32')
df_merged['voids_entropy'] = df_merged['voids_entropy'].astype('float32')
df_merged['connected_components_amplitude'] = df_merged['connected_components_amplitude'].astype('float32')
df_merged['loops_amplitude'] = df_merged['loops_amplitude'].astype('float32')
df_merged['voids_amplitude'] = df_merged['voids_amplitude'].astype('float32')
df_merged['connected_components_number_of_points'] = df_merged['connected_components_number_of_points'].astype('float32')
df_merged['loops_number_of_points'] = df_merged['loops_number_of_points'].astype('float32')
df_merged['voids_number_of_points'] = df_merged['voids_number_of_points'].astype('float32')

In [7]:
df_merged.columns

Index(['date', 'open', 'high', 'low', 'close', 'volume', 'original_close',
       'processed_log_return', 'outliers_processed_log_return',
       'normalized_outliers_processed_log_return',
       'processed_log_return_wtmra_0', 'processed_log_return_wtmra_1',
       'processed_log_return_wtmra_2', 'processed_log_return_wtmra_3',
       'processed_log_return_wtmra_4', 'processed_log_return_wtmra_5',
       'processed_log_return_wtmra_5_4', 'processed_log_return_wtmra_5_4_3',
       'processed_log_return_wtmra_5_4_3_2',
       'processed_log_return_wtmra_5_4_3_2_1',
       'processed_log_return_wtmra_5_4_3_2_1_0',
       'processed_log_return_wtmra_0_1', 'processed_log_return_wtmra_0_1_2',
       'processed_log_return_wtmra_0_1_2_3',
       'processed_log_return_wtmra_0_1_2_3_4',
       'processed_log_return_wtmra_0_1_2_3_4_5', 'initial_slice_position',
       'slice_size', 'start_date', 'connected_components_entropy',
       'loops_entropy', 'voids_entropy', 'connected_components_ampli

In [8]:
# Filter df_merged by a <= certain date
df_merged = df_merged[df_merged['date'] <= '2020-06-30'] # 2022-12-31, 2020-06-30, 2022-04-30

## Create the `series` object

In [9]:
# Convert to TimeSeries object
series_log_return = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=[target_feature])
# series_log_return = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['processed_log_return_wtmra_5_4_3_2_1'])
series_connected_components_entropy = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['connected_components_entropy'])
series_loops_entropy = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['loops_entropy'])
series_voids_entropy = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['voids_entropy'])
series_connected_components_amplitude = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['connected_components_amplitude'])
series_loops_amplitude = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['loops_amplitude'])
series_voids_amplitude = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['voids_amplitude'])
series_connected_components_number_of_points = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['connected_components_number_of_points'])
series_loops_number_of_points = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['loops_number_of_points'])
series_voids_number_of_points = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['voids_number_of_points'])

## Split the data into train, validation and test sets

In [10]:
train_date_cut = "20200430" # 20221031, 20200430, 20220228
val_date_cut = "20200531" # 20221130, 20200531, 20220331

train_log_return, temp_log_return = series_log_return.split_after(pd.Timestamp(train_date_cut))
val_log_return, test_log_return = temp_log_return.split_after(pd.Timestamp(val_date_cut))

train_connected_components_entropy, temp_connected_components_entropy = series_connected_components_entropy.split_after(pd.Timestamp(train_date_cut))
val_connected_components_entropy, test_connected_components_entropy = temp_connected_components_entropy.split_after(pd.Timestamp(val_date_cut))

train_loops_entropy, temp_loops_entropy = series_loops_entropy.split_after(pd.Timestamp(train_date_cut))
val_loops_entropy, test_loops_entropy = temp_loops_entropy.split_after(pd.Timestamp(val_date_cut))

train_voids_entropy, temp_voids_entropy = series_voids_entropy.split_after(pd.Timestamp(train_date_cut))
val_voids_entropy, test_voids_entropy = temp_voids_entropy.split_after(pd.Timestamp(val_date_cut))

train_connected_components_amplitude, temp_connected_components_amplitude = series_connected_components_amplitude.split_after(pd.Timestamp(train_date_cut))
val_connected_components_amplitude, test_connected_components_amplitude = temp_connected_components_amplitude.split_after(pd.Timestamp(val_date_cut))

train_loops_amplitude, temp_loops_amplitude = series_loops_amplitude.split_after(pd.Timestamp(train_date_cut))
val_loops_amplitude, test_loops_amplitude = temp_loops_amplitude.split_after(pd.Timestamp(val_date_cut))

train_voids_amplitude, temp_voids_amplitude = series_voids_amplitude.split_after(pd.Timestamp(train_date_cut))
val_voids_amplitude, test_voids_amplitude = temp_voids_amplitude.split_after(pd.Timestamp(val_date_cut))

train_connected_components_number_of_points, temp_connected_components_number_of_points = series_connected_components_number_of_points.split_after(pd.Timestamp(train_date_cut))
val_connected_components_number_of_points, test_connected_components_number_of_points = temp_connected_components_number_of_points.split_after(pd.Timestamp(val_date_cut))

train_loops_number_of_points, temp_loops_number_of_points = series_loops_number_of_points.split_after(pd.Timestamp(train_date_cut))
val_loops_number_of_points, test_loops_number_of_points = temp_loops_number_of_points.split_after(pd.Timestamp(val_date_cut))

train_voids_number_of_points, temp_voids_number_of_points = series_voids_number_of_points.split_after(pd.Timestamp(train_date_cut))
val_voids_number_of_points, test_voids_number_of_points = temp_voids_number_of_points.split_after(pd.Timestamp(val_date_cut))

In [11]:
# train_log_return.plot(label="train")
# val_log_return.plot(label="val")
# test_log_return.plot(label="test")

# Make the train, val and test sets plot using plotly go


fig = go.Figure()

fig.add_trace(go.Scatter(x=train_log_return.time_index, y=train_log_return.pd_series(), name='train'))
fig.add_trace(go.Scatter(x=val_log_return.time_index, y=val_log_return.pd_series(), line=dict(color="red"), name='val'))
fig.add_trace(go.Scatter(x=test_log_return.time_index, y=test_log_return.pd_series(), line=dict(color="red"), name='val'))

fig.update_layout(title=f'Train, val and test sets for feature: {target_feature}')

fig.show()

## Scale the data

In [12]:
# Normalize
scaler_connected_components_entropy = Scaler()
scaler_loops_entropy = Scaler()
scaler_voids_entropy = Scaler()
scaler_connected_components_amplitude = Scaler()
scaler_loops_amplitude = Scaler()
scaler_voids_amplitude = Scaler()
scaler_connected_components_number_of_points = Scaler()
scaler_loops_number_of_points = Scaler()
scaler_voids_number_of_points = Scaler()

train_connected_components_entropy_scaled = scaler_connected_components_entropy.fit_transform(train_connected_components_entropy)
val_connected_components_entropy_scaled = scaler_connected_components_entropy.transform(val_connected_components_entropy)
test_connected_components_entropy_scaled = scaler_connected_components_entropy.transform(test_connected_components_entropy)

train_loops_entropy_scaled = scaler_loops_entropy.fit_transform(train_loops_entropy)
val_loops_entropy_scaled = scaler_loops_entropy.transform(val_loops_entropy)
test_loops_entropy_scaled = scaler_loops_entropy.transform(test_loops_entropy)

train_voids_entropy_scaled = scaler_voids_entropy.fit_transform(train_voids_entropy)
val_voids_entropy_scaled = scaler_voids_entropy.transform(val_voids_entropy)
test_voids_entropy_scaled = scaler_voids_entropy.transform(test_voids_entropy)

train_connected_components_amplitude_scaled = scaler_connected_components_amplitude.fit_transform(train_connected_components_amplitude)
val_connected_components_amplitude_scaled = scaler_connected_components_amplitude.transform(val_connected_components_amplitude)
test_connected_components_amplitude_scaled = scaler_connected_components_amplitude.transform(test_connected_components_amplitude)

train_loops_amplitude_scaled = scaler_loops_amplitude.fit_transform(train_loops_amplitude)
val_loops_amplitude_scaled = scaler_loops_amplitude.transform(val_loops_amplitude)
test_loops_amplitude_scaled = scaler_loops_amplitude.transform(test_loops_amplitude)

train_voids_amplitude_scaled = scaler_voids_amplitude.fit_transform(train_voids_amplitude)
val_voids_amplitude_scaled = scaler_voids_amplitude.transform(val_voids_amplitude)
test_voids_amplitude_scaled = scaler_voids_amplitude.transform(test_voids_amplitude)

train_connected_components_number_of_points_scaled = scaler_connected_components_number_of_points.fit_transform(train_connected_components_number_of_points)
val_connected_components_number_of_points_scaled = scaler_connected_components_number_of_points.transform(val_connected_components_number_of_points)
test_connected_components_number_of_points_scaled = scaler_connected_components_number_of_points.transform(test_connected_components_number_of_points)

train_loops_number_of_points_scaled = scaler_loops_number_of_points.fit_transform(train_loops_number_of_points)
val_loops_number_of_points_scaled = scaler_loops_number_of_points.transform(val_loops_number_of_points)
test_loops_number_of_points_scaled = scaler_loops_number_of_points.transform(test_loops_number_of_points)

train_voids_number_of_points_scaled = scaler_voids_number_of_points.fit_transform(train_voids_number_of_points)
val_voids_number_of_points_scaled = scaler_voids_number_of_points.transform(val_voids_number_of_points)
test_voids_number_of_points_scaled = scaler_voids_number_of_points.transform(test_voids_number_of_points)

## Create `past_covariates`

In [13]:
train_past_covariates = concatenate([train_connected_components_entropy_scaled, train_loops_entropy_scaled, train_voids_entropy_scaled, train_connected_components_amplitude_scaled, train_loops_amplitude_scaled, train_voids_amplitude_scaled, train_connected_components_number_of_points_scaled, train_loops_number_of_points_scaled, train_voids_number_of_points_scaled], axis=1)
val_past_covariates = concatenate([val_connected_components_entropy_scaled, val_loops_entropy_scaled, val_voids_entropy_scaled, val_connected_components_amplitude_scaled, val_loops_amplitude_scaled, val_voids_amplitude_scaled, val_connected_components_number_of_points_scaled, val_loops_number_of_points_scaled, val_voids_number_of_points_scaled], axis=1)
test_past_covariates = concatenate([test_connected_components_entropy_scaled, test_loops_entropy_scaled, test_voids_entropy_scaled, test_connected_components_amplitude_scaled, test_loops_amplitude_scaled, test_voids_amplitude_scaled, test_connected_components_number_of_points_scaled, test_loops_number_of_points_scaled, test_voids_number_of_points_scaled], axis=1)
past_covariates = concatenate([train_past_covariates, val_past_covariates, test_past_covariates], axis=0)

## Training and validade Univariate model

In [14]:
window_size = 7 
num_forecast_points = 1
df_metrics = pd.DataFrame()

In [15]:
uni_model_nbeats = NBEATSModel(
    input_chunk_length=window_size,
    output_chunk_length=num_forecast_points,
    batch_size= 2 * (window_size + num_forecast_points),
    random_state=0,
    n_epochs=100,
    num_layers=2,
    layer_widths=512,
    loss_fn=torch.nn.MSELoss(),
)

In [16]:
uni_model_nbeats.fit(train_log_return,
                    val_series=val_log_return, 
                    verbose=False)

NBEATSModel(output_chunk_shift=0, generic_architecture=True, num_stacks=30, num_blocks=1, num_layers=2, layer_widths=512, expansion_coefficient_dim=5, trend_polynomial_degree=2, dropout=0.0, activation=ReLU, input_chunk_length=7, output_chunk_length=1, batch_size=16, random_state=0, n_epochs=100, loss_fn=MSELoss())

In [17]:
# concatenate test and val series
test_val_log_return = concatenate([val_log_return, test_log_return], axis=0)

In [18]:
pred = uni_model_nbeats.predict(n=(len(val_log_return) + len(test_log_return)))

fig = go.Figure()

fig.add_trace(go.Scatter(x=train_log_return.time_index, y=train_log_return.pd_series(), name='train'))
fig.add_trace(go.Scatter(x=test_val_log_return.time_index, y=test_val_log_return.pd_series(), name='test'))
fig.add_trace(go.Scatter(x=pred.time_index, y=pred.pd_series(), name='forecast'))

# add title with R2, MAPE, MAE and RMSE
fig.update_layout(title="Univariate Results<br>Evaluation Metrics: R2: {}\n".format(r2_score(series_log_return, pred))
    + " MAPE: {}".format(mape(series_log_return, pred))
    + " MAE: {}".format(mae(series_log_return, pred))
    + " RMSE: {}".format(rmse(series_log_return, pred)))

fig.show()

# series_log_return.plot(label="actual")
# pred.plot(label="forecast")
# plt.legend()
# print("MAPE = {:.2f}%".format(mape(series_log_return, pred)))
# print("MAE = {:.5f}".format(mae(series_log_return, pred)))
# print("RMSE = {:.5f}".format(rmse(series_log_return, pred)))
# print("R2 = {:.2f}".format(r2_score(series_log_return, pred)))

Predicting: |          | 0/? [00:00<?, ?it/s]

In [19]:
# uni_pred_series_log_return = uni_model_nbeats.historical_forecasts(
#     test_log_return,
#     forecast_horizon=num_forecast_points,
#     stride=1,
#     retrain=False,
#     verbose=False,
# )

In [20]:
# display_forecast_plotly("Univariate", uni_pred_series_log_return, test_log_return)

In [21]:
# Export a .json with the CCY, scenario, R2, MAPE, MAE and RMSE
dict_metrics = {
    "CCY": data_name,
    "scenario": "TDA",
    "type": "univariate",
    "R2": r2_score(series_log_return, pred),
    "MAPE": mape(series_log_return, pred),
    "MAE": mae(series_log_return, pred),
    "RMSE": rmse(series_log_return, pred)
}

# Append a dict to the df_metrics
df_metrics = df_metrics._append(dict_metrics, ignore_index=True)

## Training and validade Multivariate model

In [22]:
multi_model_nbeats = NBEATSModel(
    input_chunk_length=window_size,
    output_chunk_length=num_forecast_points,
    batch_size= 2 * (window_size + num_forecast_points),
    random_state=0,
    n_epochs=100,
    num_layers=2,
    layer_widths=512,
    loss_fn=torch.nn.MSELoss(),
)

In [23]:
# fit using multiple (two) target series
multi_model_nbeats.fit(train_log_return,
          val_series=val_log_return,
          past_covariates=train_past_covariates,
          val_past_covariates=val_past_covariates,
          verbose=False
          )

NBEATSModel(output_chunk_shift=0, generic_architecture=True, num_stacks=30, num_blocks=1, num_layers=2, layer_widths=512, expansion_coefficient_dim=5, trend_polynomial_degree=2, dropout=0.0, activation=ReLU, input_chunk_length=7, output_chunk_length=1, batch_size=16, random_state=0, n_epochs=100, loss_fn=MSELoss())

In [24]:
pred = multi_model_nbeats.predict(n=(len(val_log_return) + len(test_log_return)), series=train_log_return, past_covariates=past_covariates)

fig = go.Figure()

fig.add_trace(go.Scatter(x=train_log_return.time_index, y=train_log_return.pd_series(), name='train'))
fig.add_trace(go.Scatter(x=test_val_log_return.time_index, y=test_val_log_return.pd_series(), name='test'))
fig.add_trace(go.Scatter(x=pred.time_index, y=pred.pd_series(), name='forecast'))

# add title with R2, MAPE, MAE and RMSE
fig.update_layout(title="Multivariate Results<br>Evaluation Metrics: R2: {}\n".format(r2_score(series_log_return, pred))
    + " MAPE: {}".format(mape(series_log_return, pred))
    + " MAE: {}".format(mae(series_log_return, pred))
    + " RMSE: {}".format(rmse(series_log_return, pred)))

fig.show()

# series_log_return.plot(label="actual")
# pred.plot(label="forecast")
# plt.legend()
# print("MAPE = {:.2f}%".format(mape(series_log_return, pred)))
# print("MAE = {:.5f}".format(mae(series_log_return, pred)))
# print("RMSE = {:.5f}".format(rmse(series_log_return, pred)))
# print("R2 = {:.2f}".format(r2_score(series_log_return, pred)))

Predicting: |          | 0/? [00:00<?, ?it/s]

In [25]:
# multi_pred_series_log_return = multi_model_nbeats.historical_forecasts(
#     test_log_return,
#     past_covariates=test_past_covariates,
#     forecast_horizon=num_forecast_points,
#     stride=1,
#     retrain=False,
#     verbose=False,
# )

In [26]:
# display_forecast_plotly("Multivariate", multi_pred_series_log_return, test_log_return)

In [27]:
# Export a .json with the CCY, scenario, R2, MAPE, MAE and RMSE
dict_metrics = {
    "CCY": data_name,
    "scenario": "TDA",
    "type": "multivariate",
    "R2": r2_score(series_log_return, pred),
    "MAPE": mape(series_log_return, pred),
    "MAE": mae(series_log_return, pred),
    "RMSE": rmse(series_log_return, pred)
}

# Append a dict to the df_metrics
df_metrics = df_metrics._append(dict_metrics, ignore_index=True)

In [28]:
df_metrics.to_csv(f'outputs/04-02_time_delay_{time_delay}_slice_size_{slice_size}-output-metrics.csv', index=False)